# 02B OrderBot with Tool Calling: From Chatbot to Simple Agent

This notebook extends **02 Pizza Order Chatbot Demo**. In notebook 02, the OrderBot kept the whole menu in its prompt and **did the price math itself**. That works most of the time, but language models are not calculators: they predict likely text, so a total can come out wrong.

In this notebook we give the model **tools**: ordinary Python functions it can ask our program to run.

| Tool | What it does |
|---|---|
| `get_menu` | Looks up items and prices from data, so the model never guesses |
| `calculate_order_total` | Calculates the total exactly, in code |
| `place_order` | Saves the order, only after the customer confirms |

**Key idea:** the model never runs code itself. It *asks* for a tool by name with arguments; **our code** runs the function and sends back the result. A model + tools + a loop is the basic design of an **AI agent** (we go deeper in Module 10).

**Before you start:** your `OPENAI_API_KEY` must be in the `.env` file in the parent folder (same as notebook 02). The model is OpenAI's `gpt-5.4-mini`.

## 0. Install Gradio (once)

The chat app uses **Gradio**. If you already ran the install cell in notebook 02, skip this step. Otherwise run the cell below **once**: it installs Gradio with exact versions that work with AI500Env. When it finishes, **restart the kernel** (Kernel > Restart), then continue with Setup.

In [ ]:
# Run once (skip if you already ran the install cell in notebook 02), then restart the kernel.
%pip install gradio==6.15.1 gradio_client==2.5.0 fastapi==0.128.1 starlette==0.50.0 annotated-doc==0.0.6 groovy==0.1.2 hf-gradio==0.4.1 pydub==0.25.1 python-multipart==0.0.33 safehttpx==0.1.7 semantic-version==2.10.0 tomlkit==0.14.0

## 1. Setup

This cell loads your API key from the `.env` file and connects to OpenAI. We use `gpt-5.4-mini`, which supports tool calling.

In [ ]:
import json
import os
from datetime import datetime

from dotenv import load_dotenv
from openai import OpenAI

# The API key lives in a .env file in the parent folder, never in the notebook itself.
load_dotenv(dotenv_path="../.env", override=True)

# Connect to OpenAI (uses OPENAI_API_KEY from .env)
client = OpenAI()
MODEL = "gpt-5.4-mini"   # any current OpenAI model that supports tool calling

## 2. The menu is data, not prompt text

Instead of pasting the menu into the system prompt, we store it as a Python dictionary. The tools read prices from here, so a price change means editing one line of data.

In [ ]:
# The menu is data in our program, not text in the prompt. The tools below read from it.
MENU = {
    "pizza": {
        "pepperoni pizza": {"large": 12.95, "medium": 10.00, "small": 7.00},
        "cheese pizza": {"large": 10.95, "medium": 9.25, "small": 6.50},
        "eggplant pizza": {"large": 11.95, "medium": 9.75, "small": 6.75},
    },
    "side": {
        "fries": {"large": 4.50, "small": 3.50},
        "greek salad": {"regular": 7.25},
    },
    "drink": {
        "coke": {"large": 3.00, "medium": 2.00, "small": 1.00},
        "sprite": {"large": 3.00, "medium": 2.00, "small": 1.00},
        "bottled water": {"regular": 5.00},
    },
}
TOPPINGS = {"extra cheese": 2.00, "mushrooms": 1.50, "sausage": 3.00,
            "canadian bacon": 3.50, "ai sauce": 1.50, "peppers": 1.00}
DELIVERY_FEE = 5.00

# The last price quote, and whether the customer has seen it. place_order checks this (see Tool 3).
QUOTE = {"result": None, "shown_to_customer": False}

## 3. The tools are ordinary Python functions

Nothing here involves AI. Two design choices are worth noticing:

- `calculate_order_total` returns a helpful **error message** for unknown items or sizes. The model reads that message and asks the customer to clarify.
- `place_order` has a **guardrail written in code**: it refuses to place an order unless the customer has already seen that exact total in an earlier reply. The system prompt also tells the model to wait for confirmation, but when we tested this notebook, a model sometimes placed the order one turn early anyway. Rules that protect money or real-world actions belong in code, where the model cannot skip them.

In [ ]:
# Tool 1: look up the menu
def get_menu():
    """Return the full menu with prices."""
    return {"menu": MENU, "toppings_per_pizza": TOPPINGS, "delivery_fee": DELIVERY_FEE}


# Tool 2: calculate the order total exactly, in code
def calculate_order_total(items, delivery=False):
    """Price each item from the menu and add the delivery fee. Returns an error for unknown items."""
    lines, subtotal = [], 0.0
    for item in items:
        name = item.get("name", "").lower().strip()
        size = item.get("size", "regular").lower()
        qty = int(item.get("quantity", 1))
        # Find the item by name in any category, so a missing or wrong category is not a problem
        sizes = next((items_[name] for items_ in MENU.values() if name in items_), None)
        if sizes is None:
            return {"error": f"'{item.get('name')}' is not on the menu. Call get_menu to see the options."}
        if size not in sizes:
            return {"error": f"{name} comes in: {', '.join(sizes)}. Ask the customer which size."}
        price = sizes[size]
        extras = [t.lower() for t in item.get("toppings", [])]
        for t in extras:
            if t not in TOPPINGS:
                return {"error": f"'{t}' is not an available topping."}
            price += TOPPINGS[t]
        line_total = round(price * qty, 2)
        subtotal += line_total
        lines.append({"item": f"{qty} x {size} {name}" + (f" + {', '.join(extras)}" if extras else ""),
                      "line_total": line_total})
    fee = DELIVERY_FEE if delivery else 0.0
    result = {"lines": lines, "subtotal": round(subtotal, 2), "delivery_fee": fee,
              "total": round(subtotal + fee, 2)}
    QUOTE.update(result=result, shown_to_customer=False)   # a new quote the customer has not seen yet
    return result


# Tool 3: place the order (here we save it to a file; a real app would send it to the kitchen)
def place_order(items, delivery, customer_name, address=""):
    """Save the confirmed order and return a confirmation number."""
    # Guardrail in CODE, not just in the prompt: the customer must have seen this exact
    # total in an earlier reply (and then answered) before the order can be placed.
    quote = QUOTE["result"]
    if quote is None or not QUOTE["shown_to_customer"]:
        return {"error": "Not placed. First show the customer the order summary and total, then wait for them to confirm."}
    priced = calculate_order_total(items, delivery)
    if "error" in priced:
        return priced
    if priced["total"] != quote["total"]:
        return {"error": "Not placed. The order changed since the customer saw the total. Show the new total and ask again."}
    QUOTE["shown_to_customer"] = True   # recalculating the same order does not hide the quote
    order = {"order_id": datetime.now().strftime("%H%M%S"), "customer_name": customer_name,
             "delivery": delivery, "address": address, **priced}
    with open("orders.json", "a", encoding="utf-8") as f:
        f.write(json.dumps(order) + "\n")
    return {"status": "placed", "order_id": order["order_id"], "total": order["total"]}

### Test the tools directly (no model involved)

Always test tools on their own first. Expected total below: 12.95 + 1.50 (mushrooms) + 1.00 + 5.00 delivery = **20.45**.

In [ ]:
order = [
    {"name": "pepperoni pizza", "size": "large", "quantity": 1, "toppings": ["mushrooms"]},
    {"name": "coke", "size": "small", "quantity": 1},
]
print(calculate_order_total(order, delivery=True))
print(calculate_order_total([{"name": "hawaiian pizza", "size": "large", "quantity": 1}]))

## 4. Describe the tools to the model

The model cannot see our Python code. It only sees each tool's **name, description, and parameters** (written in JSON Schema). Clear descriptions are what make the model call the right tool at the right time, so treat them like part of your prompt. The system prompt also tells the model to use the tools instead of guessing.

In [ ]:
# These descriptions are what the model reads to decide when and how to call each tool.
ITEM_SCHEMA = {
    "type": "object",
    "properties": {
        "name": {"type": "string", "description": "Item name exactly as on the menu, e.g. 'cheese pizza'"},
        "size": {"type": "string", "description": "small, medium, large, or regular"},
        "quantity": {"type": "integer"},
        "toppings": {"type": "array", "items": {"type": "string"}, "description": "Extra toppings (pizza only)"},
    },
    "required": ["name", "size", "quantity"],
}

TOOLS = [
    {"type": "function", "function": {
        "name": "get_menu",
        "description": "Get the restaurant menu with all items, sizes, toppings, and prices.",
        "parameters": {"type": "object", "properties": {}},
    }},
    {"type": "function", "function": {
        "name": "calculate_order_total",
        "description": "Calculate the exact price of an order. Always use this instead of doing math yourself.",
        "parameters": {"type": "object", "properties": {
            "items": {"type": "array", "items": ITEM_SCHEMA},
            "delivery": {"type": "boolean", "description": "True if the order is for delivery"},
        }, "required": ["items", "delivery"]},
    }},
    {"type": "function", "function": {
        "name": "place_order",
        "description": "Place the order. Only call this after the customer has confirmed the summary and total.",
        "parameters": {"type": "object", "properties": {
            "items": {"type": "array", "items": ITEM_SCHEMA},
            "delivery": {"type": "boolean"},
            "customer_name": {"type": "string"},
            "address": {"type": "string", "description": "Delivery address, if delivery"},
        }, "required": ["items", "delivery", "customer_name"]},
    }},
]

FUNCTIONS = {"get_menu": get_menu, "calculate_order_total": calculate_order_total, "place_order": place_order}

SYSTEM_PROMPT = """You are OrderBot, a friendly assistant that takes orders for a pizza restaurant.
Greet the customer, collect the order, and ask whether it is for pickup or delivery.
Use the get_menu tool to answer questions about the menu; do not guess items or prices.
Use the calculate_order_total tool for every price or total; never do the math yourself.
Clarify sizes and toppings so each item matches the menu exactly.
Before placing the order, summarize it with the total and ask the customer to confirm.
For delivery, ask for an address. Ask for the customer's name.
Only call place_order after the customer confirms. Then give them the order number.
Keep replies short and conversational."""

## 5. The agent loop

This is the heart of tool calling:

1. Send the conversation **and the tool definitions** to the model.
2. If the model replies with text, we are done: show it to the customer.
3. If the model replies with **tool calls**, run each function in our code and add the results to the conversation.
4. Go back to step 1, so the model can use the results (it may call more tools).

The `tool_log` list records every call so we can see what happened. When the model replies with text, the loop also marks the latest price quote as *shown to the customer*, which is what allows `place_order` to proceed on a later turn.

In [ ]:
from difflib import SequenceMatcher

tool_log = []   # every tool call, so we can see what the model did


def keep_final_copy(text):
    """gpt-5.4-mini sometimes writes a short draft of its reply and then the final reply, joined together.
    If the text is two near-identical copies, keep only the final one."""
    lines = (text or "").strip().split("\n")
    for k in range(1, len(lines)):
        first, second = "\n".join(lines[:k]).strip(), "\n".join(lines[k:]).strip()
        if first and second and SequenceMatcher(None, first, second).ratio() > 0.8:
            return second
    return text


def run_turn(messages):
    """Send the conversation to the model. If it asks for tools, run them and send back
    the results, repeating until the model replies with text for the customer."""
    while True:
        response = client.chat.completions.create(model=MODEL, messages=messages, tools=TOOLS)
        msg = response.choices[0].message

        if not msg.tool_calls:                       # no tool needed: this is the reply
            reply = keep_final_copy(msg.content)
            messages.append({"role": "assistant", "content": reply})
            if QUOTE["result"] is not None:
                QUOTE["shown_to_customer"] = True    # the customer now sees the latest quote
            return reply

        # The model asked for one or more tools. Record its request...
        messages.append({"role": "assistant", "content": msg.content or "",
                         "tool_calls": [tc.model_dump() for tc in msg.tool_calls]})
        # ...run each tool in OUR code, and send each result back.
        for call in msg.tool_calls:
            args = json.loads(call.function.arguments or "{}")
            result = FUNCTIONS[call.function.name](**args)
            tool_log.append(f"{call.function.name}({json.dumps(args)}) -> {json.dumps(result)}")
            messages.append({"role": "tool", "tool_call_id": call.id, "content": json.dumps(result)})

## 6. Try a conversation in text first

Before building a user interface, test with a scripted conversation and inspect the tool log.

In [ ]:
messages = [{"role": "system", "content": SYSTEM_PROMPT}]

for user_text in ["Hi! I'd like a large pepperoni pizza with mushrooms and a small coke.",
                  "Delivery please, to 12 Main Street. My name is Pat.",
                  "Yes, that's everything. Please place the order."]:
    messages.append({"role": "user", "content": user_text})
    print("USER:", user_text)
    print("BOT :", run_turn(messages), "\n")

print("TOOL CALLS:")
for line in tool_log:
    print(" -", line[:200])

## 7. Chat with OrderBot in a web interface (Gradio)

The same interface as notebook 02, plus a panel showing which tools the model called on each turn and what our code returned. Click the link that appears to open it. **New order** clears the chat and starts over; **Quit** stops the app.

If you see `No module named 'gradio'`, run the install step (section 0), restart the kernel, and run the notebook again. Section 6 above works without Gradio.

> **Open the public link.** The cell prints two links. Click the `https://….gradio.live` link (next to *Running on public URL*). The local `http://127.0.0.1:…` link points to the HPC server itself, not to your computer, so it gives an error in your browser. The chat window shown below the cell may also stay blank on the HPC; use the public link.

> **Keep your app link private.** The `https://….gradio.live` link works for anyone who has it, for about a week, and every message is charged to your API key. Do not post it in discussions or screenshots, and click **Quit** when you are done: it stops the app and the link.

In [ ]:
import threading
import gradio as gr


def respond(user_message, chat_history, conversation):
    # `conversation` is the full message list sent to the model, including tool calls and results;
    # `chat_history` is only what the customer sees.
    if not conversation:
        conversation = [{"role": "system", "content": SYSTEM_PROMPT}]
    conversation.append({"role": "user", "content": user_message})

    calls_before = len(tool_log)
    try:
        reply = run_turn(conversation)
    except Exception as e:                               # show the problem (for example, a missing API key)
        conversation.pop()
        return user_message, chat_history + [{"role": "assistant", "content": f"⚠️ {type(e).__name__}: {e}"}], conversation, ""
    new_calls = tool_log[calls_before:]

    chat_history = chat_history + [{"role": "user", "content": user_message},
                                   {"role": "assistant", "content": reply}]
    tools_used = "\n\n".join(new_calls) if new_calls else "(no tools used this turn)"
    return "", chat_history, conversation, tools_used


def new_order():
    QUOTE.update(result=None, shown_to_customer=False)   # forget any price quote from the last order
    return "", [], [], ""


def quit_app(chat_history):
    threading.Timer(1, demo.close).start()               # stop the app a second after the goodbye message shows
    chat_history = chat_history + [{"role": "assistant", "content": "OrderBot has stopped. Go back to the notebook."}]
    off = gr.Button(interactive=False)
    return gr.Textbox(value="", interactive=False), chat_history, off, off, off


with gr.Blocks(title="OrderBot with Tools") as demo:
    gr.Markdown("# 🍕 OrderBot with Tools")
    gr.Markdown("Order a pizza. On the right, watch which tools the model calls and what our code returns.")
    conversation = gr.State([])
    with gr.Row():
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(height=500)
            msg = gr.Textbox(placeholder="Enter text here…", label="Your message")
            with gr.Row():
                submit_btn = gr.Button("Chat!")
                new_order_btn = gr.Button("New order")
                quit_btn = gr.Button("Quit")
        with gr.Column(scale=2):
            tool_box = gr.Textbox(label="Tool calls this turn", lines=22, interactive=False)

    msg.submit(respond, inputs=[msg, chatbot, conversation], outputs=[msg, chatbot, conversation, tool_box])
    submit_btn.click(respond, inputs=[msg, chatbot, conversation], outputs=[msg, chatbot, conversation, tool_box])
    new_order_btn.click(new_order, outputs=[msg, chatbot, conversation, tool_box])
    quit_btn.click(quit_app, inputs=[chatbot], outputs=[msg, chatbot, submit_btn, new_order_btn, quit_btn])

demo.launch(share=True)

## 8. The customer's view

In section 7 we looked at the bot as its **builders**: the tool panel shows every call. A real customer sees none of
that, only the chat. The cell below opens the **same bot** (the same tools, agent loop and rules) the way a customer
would see it: a greeting, a chat window and a message box. This is how you would show your bot to someone else, for
example in a project demo.

The same notes as section 7 apply: open the `https://….gradio.live` link, keep it private (every message is charged to
your API key), and click **Close** when you are done.

> This simple version keeps one price quote at a time (`QUOTE`), so it is meant for **one customer at a time**. A real
> app gives every customer their own conversation and quote.

In [ ]:
# The same OrderBot as a customer sees it: only the chat. No tool panel, nothing to edit.
GREETING = "Hi! Welcome to our pizza restaurant. What can I get for you today?"


def customer_respond(user_message, chat_history, conversation):
    if not conversation:                                 # a new customer: start a fresh order
        QUOTE.update(result=None, shown_to_customer=False)
        conversation = [{"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "assistant", "content": GREETING}]
    conversation.append({"role": "user", "content": user_message})
    try:
        reply = run_turn(conversation)                   # the same agent loop and tools as above
    except Exception:
        conversation.pop()
        reply = "Sorry, something went wrong. Please try again."
    chat_history = chat_history + [{"role": "user", "content": user_message},
                                   {"role": "assistant", "content": reply}]
    return "", chat_history, conversation


with gr.Blocks(title="Order pizza") as customer_app:
    gr.Markdown("# 🍕 Order pizza online\nPizza, sides and drinks, for pickup or delivery.")
    customer_conversation = gr.State([])
    customer_chat = gr.Chatbot(value=[{"role": "assistant", "content": GREETING}], height=450, show_label=False)
    customer_msg = gr.Textbox(placeholder="Type your order and press Enter", show_label=False)
    with gr.Row():
        send_btn = gr.Button("Send", variant="primary")
        close_btn = gr.Button("Close")

    chat_parts = [customer_msg, customer_chat, customer_conversation]
    customer_msg.submit(customer_respond, inputs=chat_parts, outputs=chat_parts)
    send_btn.click(customer_respond, inputs=chat_parts, outputs=chat_parts)
    close_btn.click(lambda: threading.Timer(1, customer_app.close).start())   # stop the app and its link

customer_app.launch(share=True)

## 9. Explore

1. **Watch the tool panel.** Ask "What pizzas do you have?" Which tool does the model call?
2. **Break it on purpose.** Order a "hawaiian pizza" or a "medium greek salad". How does the error message from our code shape the model's reply?
3. **Remove a prompt rule.** In `SYSTEM_PROMPT`, delete the line that says to use `calculate_order_total` for every price. Does the model still call it? Why does the tool *description* matter too?
4. **Test the code guardrail.** Say "Large cheese pizza for pickup, I'm Sam, place it now." Watch the tool panel: what happens when the model tries to place the order before you have seen the total?
5. **Add a tool.** Write a `get_wait_time(delivery)` function (for example, 20 minutes for pickup and 45 for delivery), add its definition to `TOOLS` and `FUNCTIONS`, and ask the bot how long your order will take.

> **Safety note:** `place_order` takes an action in the real world (in a real restaurant it would charge a card and start cooking). That is why both the system prompt and the code require the customer to confirm first. Deciding which actions need human confirmation is one of the most important design choices when building agents.